# **BART and BCF: Bayesian trees for the effect**

Practice for the module [“Heterogeneous Effects”](https://ai-interpretability.school/learn/5/0/0), lesson “Bayesian trees: BART and BCF”.

In the lesson “Bayesian trees: BART and BCF” BART ([Chipman, George, and McCulloch, 2010](https://doi.org/10.1214/09-AOAS285)) was a sum of many small trees with a prior distribution, and [Hill (2011)](https://doi.org/10.1198/jcgs.2010.08162) proposed using it to estimate the effect: fit one outcome model on the features and the treatment and take the difference of predictions, as in the S-learner. The Bayesian causal forest, BCF ([Hahn, Murray, and Carvalho, 2020](https://arxiv.org/abs/1706.09523)), splits the model into two sums of trees, a prognostic one $\mu(x)$ and the effect $\tau(x)$ with its own, stricter prior, and adds a propensity estimate $\hat\pi(x)$ as a feature. Here we check three claims of the lesson:

1. the posterior distribution gives every object an interval for the effect without any extra procedure;
2. under targeted selection — when the treatment goes more often to objects with a worse outcome without it — regularization biases the effect estimate, and the feature $\hat\pi(x)$ reduces that bias;
3. separate models for the two groups, as in the T-learner, give a worse estimate than one model with an explicit effect function.

Plan of the notebook:

1. the data of Hahn et al.: a nonlinear prognosis, a heterogeneous effect, targeted selection;
2. BART with the treatment as a feature (Hill, 2011);
3. a propensity estimate and BART with it as a feature;
4. BCF and a comparison of four schemes;
5. where it breaks: bias from regularization and the intervals of one MCMC chain;
6. implementations.

The notebook runs on a CPU; the whole run takes four to six minutes, almost all of it the repetitions in section 5.1.

In [ ]:
!pip install -q stochtree==0.4.5

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from stochtree import BARTModel, BCFModel

SEED = 0
MCMC = dict(num_gfr=10, num_burnin=200, num_mcmc=500)   # 10 greedy passes to start; in every chain 200 discarded and 500 kept MCMC steps
CHAINS = 4                                               # four independent MCMC chains, 2000 samples in total
pd.set_option('display.width', 140)

## 1. The data of Hahn et al.

We follow the design of section 6.1 of the BCF paper: 500 objects, five features — three continuous $x_1,x_2,x_3\sim\mathcal N(0,1)$, a binary $x_4$ and a categorical $x_5\in\{1,2,3\}$. A nonlinear prognosis without treatment and a heterogeneous effect:

$$\mu(x)=-6+g(x_5)+6\,|x_3-1|,\qquad \tau(x)=1+2\,x_2x_5,$$

where $g(1)=2$, $g(2)=-1$, $g(3)=-4$. In the paper's text $g$ is applied to the binary $x_4$, but $g$ has three values, like the levels of $x_5$; we apply $g$ to $x_5$, and $x_4$ remains a feature nothing depends on. The propensity is targeted selection: the probability of treatment grows with $\mu(x)$,

$$\pi(x)=0.8\,\Phi\bigl(3\mu(x)/s-0.5\,x_1\bigr)+0.05+u/10,\qquad u\sim U(0,1),$$

where $s$ is the standard deviation of $\mu$ over the sample. The outcome is $Y=\mu(x)+\tau(x)\,Z+\varepsilon$, $\varepsilon\sim\mathcal N(0,1)$. The categorical $x_5$ is encoded with three indicators, and it enters $\tau$ by its number.

In [ ]:
def simulate(n, r, heterogeneous=True):
    x = r.normal(size=(n, 3))
    x4 = r.binomial(1, 0.5, n)
    x5 = r.integers(1, 4, n)
    mu = -6 + np.select([x5 == 1, x5 == 2], [2.0, -1.0], -4.0) + 6 * np.abs(x[:, 2] - 1)
    tau = 1 + 2 * x[:, 1] * x5 if heterogeneous else np.full(n, 3.0)
    pi = 0.8 * norm.cdf(3 * mu / mu.std() - 0.5 * x[:, 0]) + 0.05 + r.uniform(size=n) / 10
    Z = r.binomial(1, pi)
    Y = mu + tau * Z + r.normal(size=n)
    X = np.column_stack([x, x4, x5 == 1, x5 == 2, x5 == 3]).astype(float)
    return X, Z, Y, tau, pi, mu


X, Z, Y, tau, pi, mu = simulate(500, np.random.default_rng(SEED))
print('objects:', len(Y), '| treated:', Z.sum())
print('propensity: from', round(pi.min(), 3), 'to', round(pi.max(), 3), '| correlation of propensity with mu(x):', round(np.corrcoef(pi, mu)[0, 1], 3))
print('true average effect:', round(tau.mean(), 3), '| standard deviation of the effect:', round(tau.std(), 3))
print('naive difference of mean outcomes:', round(Y[Z == 1].mean() - Y[Z == 0].mean(), 3))

The naive difference of means is far from the true average effect: the treatment goes to objects with a larger $\mu(x)$, and the difference of groups mixes the effect with the difference in prognosis.

## 2. BART with the treatment as a feature (Hill, 2011)

One sum of trees $f(x,z)$ is fitted on the features and the treatment together. Each kept MCMC step gives one sample of the function $f$ from the posterior, and for object $i$ the effect in that sample is $f(x_i,1)-f(x_i,0)$. We run four independent chains of 500 kept steps each; over 2000 samples we get the posterior distribution of every object's effect: the mean is the point estimate, and the 2.5 and 97.5 percent quantiles give the 95 percent interval. The sample average effect is the same for the mean over objects.

In `stochtree`, the model `BARTModel().sample(X, y, ...)` returns prediction samples via `predict(X, terms='y_hat')` — an “objects $\times$ samples” matrix. `random_seed` in `general_params` fixes the MCMC generator, and `num_chains` sets the number of chains.

**Task 1. Write a function `bart_s(Xa, Z, Y)`: it fits a `BARTModel` on the features `Xa` with a treatment column and returns the matrix of effect samples: the prediction at $Z=1$ minus the prediction at $Z=0$. Write a function `summarize(draws, tau)` that returns the error of the average effect estimate (posterior mean minus the true one), whether the true average effect is covered by the 95 percent interval, the RMSE of the CATE estimate over objects (posterior means against the true $\tau(x_i)$), the share of objects whose interval covered $\tau(x_i)$, and the mean interval width. What is the RMSE of the CATE estimate for BART?**

In [ ]:
# Your code here

**Your answer:**

## 3. The propensity estimate as a feature

Hahn et al. show why BART with the treatment as a feature is biased under targeted selection. The prior pulls the function toward a simple form. If the propensity is nearly monotone in $\mu(x)$, the column $Z$ on its own predicts the outcome well: a simple model can attribute part of the difference in $\mu(x)$ to the treatment. The authors call this regularization-induced confounding (RIC). The remedy is to give the model a feature that carries this information directly: the propensity estimate $\hat\pi(x)$.

We also estimate the propensity with a sum of trees — a regression of $Z$ on $X$, as `stochtree` does inside `BCFModel` when no propensity is passed. The estimate is clipped to $[0.01, 0.99]$.

In [ ]:
def propensity(X, Z, seed=SEED):
    m = BARTModel()
    m.sample(X, Z.astype(float), num_gfr=10, num_burnin=0, num_mcmc=10, general_params={'random_seed': seed, 'num_threads': 1})
    return np.clip(m.predict(X, terms='y_hat', type='mean'), 0.01, 0.99)


pihat = propensity(X, Z)
print('correlation of the propensity estimate with the true one:', round(np.corrcoef(pihat, pi)[0, 1], 3))
draws['ps-BART'] = bart_s(np.column_stack([X, pihat]), Z, Y)
print(pd.Series(summarize(draws['ps-BART'], tau)).round(3))

## 4. BCF and a comparison of four schemes

BCF writes the outcome as $Y=\mu(x,\hat\pi(x))+\tau(x)\,Z+\varepsilon$: two sums of trees with different priors. The prognostic $\mu$ has 250 trees (the `stochtree` default; the paper by Hahn et al. uses 200) with the usual BART prior ($\eta=0.95$, $\beta=2$ in the splitting probability $\eta\,(1+d)^{-\beta}$ of a node at depth $d$). The effect $\tau$ has fewer trees and $\eta=0.25$, $\beta=3$: the trees barely grow, and the prior pulls the effect toward being the same for everyone. The paper gives $\tau$ 50 trees, `stochtree` defaults to 100; we take 50, as in the paper.

For comparison we add a fourth scheme — two separate BARTs for the treatment and control groups, an analog of the T-learner (BART $(f_0,f_1)$ in the paper).

**Task 2 (for self-check: the site has no question on it — MCMC on another processor follows a different trajectory, and the number may differ from ours: with seeds 0–4 it lies between 1.28 and 1.31). Fit a `BCFModel` with `propensity_train=pihat` and `treatment_effect_forest_params={'num_trees': 50}`; the effect samples are `predict(X, Z, propensity=pihat, terms='tau')`. Fit two separate `BARTModel`s on the objects with $Z=1$ and with $Z=0$ and take the difference of their predictions for all objects. Collect `summarize` for the four schemes in a table. What is the RMSE of the CATE estimate for the two separate BARTs?**

In [ ]:
# Your code here

**Your answer:**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name in zip(axes, ['BART', 'BCF']):
    c = draws[name].mean(axis=1)
    lo, hi = np.quantile(draws[name], [0.025, 0.975], axis=1)
    order = np.argsort(tau)
    ax.fill_between(range(len(tau)), lo[order], hi[order], color='#9ecae1', alpha=0.6, label='95% posterior interval')
    ax.plot(range(len(tau)), c[order], '.', ms=3, color='#3b6ea5', label='Posterior mean')
    ax.plot(range(len(tau)), tau[order], color='#c0392b', lw=1.5, label='True effect')
    ax.set_title(name)
    ax.set_xlabel('Objects sorted by true effect')
    ax.set_ylabel('Effect')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Where it breaks

### 5.1. Regularization-induced confounding

One sample is one draw of noise. To see the bias rather than chance, we repeat the computation on 10 samples with the same effect for everyone, $\tau(x)=3$, where RIC shows in its pure form: any deviation of the average effect estimate from 3 is an error, not heterogeneity. On each sample we fit three schemes: BART without $\hat\pi$, BCF without $\hat\pi$ (`propensity_covariate='none'`) and BCF with $\hat\pi$.

**Task 3. For the samples `simulate(500, np.random.default_rng(100 + k), heterogeneous=False)`, $k=0,\dots,9$, compute for the three schemes the mean error of the average effect estimate over samples, its RMSE, and the share of samples where the 95 percent interval of the average effect covered 3. How does the feature $\hat\pi$ change the error?** Check your answer by the number of samples out of 10 where the interval covered 3: for BART and BCF without $\hat\pi$ it does not change with MCMC seeds 0–4, while for BCF with $\hat\pi$ the mean error ranges from 0.021 to 0.036 across seeds.

In [ ]:
# Your code here

**Your answer:**

### 5.2. One MCMC chain

BART intervals are quantiles of samples from the posterior, and they are right only insofar as the MCMC chain has explored that distribution. A sum of trees is rebuilt slowly: one chain can wander around one set of trees for a long time, and then the spread of the samples is smaller than the true posterior uncertainty. Independent chains start from different greedy passes and see this.

**Task 4. Fit the BART from section 2 with one chain (`chains=1`, the same seed) and compare it with four chains: the RMSE of the CATE estimate, the share of objects whose 95 percent interval covered $\tau(x_i)$, and the mean interval width. Then, for the four chains, compute the same share of objects with a covered $\tau(x_i)$ separately from the samples of each chain (the samples of chain $c$ are the columns from $500c$ to $500(c+1)-1$). What happens to the intervals of one chain?**

In [ ]:
# Your code here

**Your answer:**

## 6. Implementations

- [`stochtree`](https://github.com/StochasticTree/stochtree) (MIT license; Python and R): `BARTModel` and `BCFModel`, as in the notebook. By default the sampler starts with several greedy passes (grow-from-root, the parameter `num_gfr`) and continues with the usual MCMC of Chipman et al.; the package's BCF uses the treatment-coding parameterization from section 5.3 of Hahn et al.
- [`pymc-bart`](https://github.com/pymc-devs/pymc-bart) (Apache-2.0 license): BART as a distribution inside a PyMC model; the effect is assembled by hand, as in section 2.
- The authors' R package `bcf` and the BART packages for R are not used in the notebook.

## What to take away

- **BART gives every object an interval for free,** but it is the model's posterior interval, not a coverage guarantee, and a single MCMC chain makes it narrower than it should be.
- **Under targeted selection regularization biases the effect estimate,** and the feature $\hat\pi(x)$ reduces that bias — for BART and for BCF.
- **Separate models for the two groups do worse than one shared model:** each has less data, and the simplicity of the effect goes unused.
- **BCF gives the effect its own prior,** so the shrinkage toward a common effect can be tuned without touching the prognostic model.

Sources:
1. [Chipman, George, and McCulloch, 2010. BART: Bayesian additive regression trees](https://doi.org/10.1214/09-AOAS285)
2. [Hill, 2011. Bayesian nonparametric modeling for causal inference](https://doi.org/10.1198/jcgs.2010.08162)
3. [Hahn, Murray, and Carvalho, 2020. Bayesian regression tree models for causal inference: regularization, confounding, and heterogeneous effects](https://arxiv.org/abs/1706.09523)
4. [stochtree](https://github.com/StochasticTree/stochtree)